# ImamAli110 — Interim Production Test 001 (GPU-independent)

**نحوه اجرا:** در Colab از منوی Runtime گزینهٔ Run all را بزن. این نوت‌بوک در هر مرحله پیام وضعیت چاپ می‌کند و در صورت خطا، پیام کامل و فایل گزارش را می‌سازد. اگر صفحه بدون خروجی است، احتمالاً هنوز سلول‌ها اجرا نشده‌اند.

این تست داستان و نریشن فارسی را آماده می‌کند؛ تصویرسازی SDXL یا آپلود یوتیوب انجام نمی‌دهد.

In [ ]:
# 1) Check runtime and prepare workspace
import os, sys, json, subprocess, traceback, importlib.util
from pathlib import Path
LOG = Path('/content/TPU_Interim_Work_001.log')
def log(msg):
    msg = str(msg); print(msg, flush=True)
    with LOG.open('a', encoding='utf-8') as f: f.write(msg + '\n')
try:
    LOG.write_text('TPU Interim Work 001 started\n', encoding='utf-8')
    log('STEP 1/5: runtime check started')
    log('Python: ' + sys.version)
    try:
        import torch
        log('PyTorch: ' + torch.__version__)
        log('CUDA available: ' + str(torch.cuda.is_available()))
    except Exception as e: log('PyTorch check warning: ' + repr(e))
    try:
        import torch_xla
        log('torch_xla available: yes')
    except Exception: log('torch_xla available: no (not required for this preparation task)')
    WORK = Path('/content/ImamAli110_interim_001'); WORK.mkdir(parents=True, exist_ok=True)
    log('Workspace: ' + str(WORK))
    log('Installing ffmpeg and Persian-capable fonts...')
    subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq ffmpeg fonts-noto-core fontconfig'], check=True)
    subprocess.run(['fc-cache','-f'], check=False)
    log('Installing edge-tts...')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','edge-tts'])
    log('STEP 1/5: OK')
except Exception:
    log('STEP 1/5: FAILED\n' + traceback.format_exc())
    raise

In [ ]:
# 2) Create one non-repetitive Persian story with a clear opening hook
log('STEP 2/5: story preparation started')
story = {
  'id': 'imamali110_interim_001',
  'title_fa': 'درختی که برای غریبه‌ها کاشته شد',
  'hook_fa': 'اگر امروز کسی از خوبی تو سود نبرد، یعنی خوبی‌ات بی‌فایده بوده؟',
  'voice': 'fa-IR-FaridNeural',
  'rate': '-8%',
  'beats': [
    {'scene': 1, 'seconds_hint': 5, 'narration': 'اگر امروز کسی از خوبی تو سود نبرد، یعنی خوبی‌ات بی‌فایده بوده؟', 'visual_prompt': 'Cinematic close-up of a tiny green sapling in dry soil at sunrise, no people, hopeful realistic photography.'},
    {'scene': 2, 'seconds_hint': 5, 'narration': 'روزی کودکی نهالی کاشت؛ با اینکه می‌دانست شاید سال‌ها طول بکشد تا سایه بدهد.', 'visual_prompt': 'A young Middle Eastern boy carefully planting a small sapling in a quiet village garden, single child, natural proportions, golden morning light.'},
    {'scene': 3, 'seconds_hint': 5, 'narration': 'مردی از او پرسید: وقتی شاید خودت زیر سایه‌اش ننشینی، چرا زحمت می‌کشی؟', 'visual_prompt': 'An elderly Middle Eastern man speaking kindly with a young boy beside a sapling, exactly two people, clear separation, realistic anatomy.'},
    {'scene': 4, 'seconds_hint': 5, 'narration': 'کودک گفت: من امروز زیر سایه‌ی درختی نشسته‌ام که کسی پیش از من کاشته بود.', 'visual_prompt': 'The boy looking up with quiet sincerity beneath the edge of a mature oak tree, one person only, natural child proportions.'},
    {'scene': 5, 'seconds_hint': 5, 'narration': 'بعضی کارهای خوب، جوابشان را از آینده می‌گیرند؛ نه از تشویق امروز.', 'visual_prompt': 'Warm sunlight filtering through the leaves of a mature tree, peaceful empty bench, cinematic realistic photography, no text.'},
    {'scene': 6, 'seconds_hint': 5, 'narration': 'پس خوبی کن؛ شاید روزی غریبه‌ای به لطفی تکیه کند که امروز تو بی‌صدا انجام دادی.', 'visual_prompt': 'A boy and an elderly man watering a small sapling together, exactly two people, hopeful golden-hour documentary photography.'}
  ],
  'cta_fa': 'اگر این پیام به دلت نشست، برای کسی بفرست که به امید نیاز دارد.',
  'safety_note': 'داستان تمثیلی است؛ به عنوان نقل‌قول تاریخی یا دینی معرفی نشود.'
}
assert story['hook_fa'] == story['beats'][0]['narration']
assert len(story['beats']) == 6 and all(len(b['narration']) > 15 for b in story['beats'])
(WORK/'storyboard.json').write_text(json.dumps(story, ensure_ascii=False, indent=2), encoding='utf-8')
print('Story QC passed:', len(story['beats']), 'scenes; hook is first narration line.')
print(story['hook_fa'])log('STEP 2/5: OK — story and storyboard.json created')

In [ ]:
# 3) Generate Persian narration segments (no GPU required)
log('STEP 3/5: Persian TTS started; requires internet access')
import asyncio, edge_tts
audio_dir = WORK/'audio'
audio_dir.mkdir(exist_ok=True)
async def make_audio():
    for beat in story['beats']:
        target = audio_dir/f"scene_{beat['scene']:02d}.mp3"
        await edge_tts.Communicate(beat['narration'], story['voice'], rate=story['rate'], pitch='-1Hz').save(str(target))
        print('Created', target.name, target.stat().st_size, 'bytes')
await make_audio()
assert len(list(audio_dir.glob('scene_*.mp3'))) == len(story['beats'])
print('TTS QC passed: all scene audio files exist.')log('STEP 3/5: OK — all audio segments created')

In [ ]:
# 4) Make a readable UTF-8 SRT draft and concatenate voice segments for listening
log('STEP 4/5: assembling narration preview and draft captions')
def srt_time(seconds):
    ms = int(round(seconds*1000)); h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); s, ms = divmod(ms, 1000)
    return f'{h:02}:{m:02}:{s:02},{ms:03}'
lines=[]; t=0.0
for i, beat in enumerate(story['beats'], 1):
    # Rough display timing only; audio durations are measured below and should replace these before final edit.
    d=float(beat['seconds_hint'])
    lines += [str(i), f'{srt_time(t)} --> {srt_time(t+d)}', beat['narration'], '']
    t += d
(WORK/'captions_draft.srt').write_text('\n'.join(lines), encoding='utf-8')
concat = WORK/'concat.txt'
concat_lines = []
for beat in story['beats']:
    audio_path = audio_dir / ('scene_%02d.mp3' % beat['scene'])
    concat_lines.append("file '" + audio_path.as_posix() + "'\n")
concat.write_text(''.join(concat_lines), encoding='utf-8')
preview = WORK/'narration_preview.mp3'
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(concat),'-c:a','libmp3lame','-q:a','3',str(preview)], check=True, capture_output=True)
print('Created:', preview)
print('Created:', WORK/'captions_draft.srt')
print('Created:', WORK/'storyboard.json')log('STEP 4/5: OK — preview and subtitle draft created')

In [ ]:
# 5) Download generated files and a diagnostic log; no upload/publishing occurs
log('STEP 5/5: bundling outputs')
from google.colab import files
import shutil
bundle = WORK/'ImamAli110_interim_001.zip'
shutil.make_archive(str(bundle.with_suffix('')), 'zip', WORK)
print('SUCCESS: output bundle ready:', bundle, flush=True)
print('Diagnostic log:', LOG, flush=True)
files.download(str(bundle))
files.download(str(LOG))
log('ALL STEPS FINISHED: SUCCESS')